# 06 — Pick and Place: Coaching Session

**Task**: Pick up the red block and place it in the bowl. Requires grasp, lift, move, and release.

**Sequence in curriculum**: Do `05_touch_objects` first. Touch builds motion consistency; pick-and-place adds grasp and precision.

**Prereqs**:
- `00_arm_setup` complete
- `05_touch_objects` trained and working (arm is consistent)
- `01_reserve_node` complete — MI100 provisioned
- Block and bowl placed in fixed, repeatable positions
- `.env` populated

**Outcome**: Trained ACT policy for pick-and-place available on HF Hub.

---

In [ ]:
import os
import re
import shlex
import time
import json
import subprocess
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from tqdm.notebook import tqdm

load_dotenv(dotenv_path=Path('..') / '.env', override=False)

_bench = {"notebook": "06_pick_place", "started_at": datetime.utcnow().isoformat(),
          "timings": {}, "config": {}}
_t0 = time.monotonic()

PI_HOST      = os.getenv("PI_HOST", "192.168.4.191")
PI_PORT      = os.getenv("PI_PORT", "22222")
PI_USER      = "root"
HF_USER      = os.getenv("HF_USER")
HF_TOKEN     = os.getenv("HF_TOKEN")
FLOATING_IP  = os.getenv("CONTROL_FLOATING_IP")
PI_IMAGE     = f"{HF_USER}/lerobot-soarm101:latest"
POLICY       = os.getenv("POLICY", "act")

# ── Task configuration ────────────────────────────────────────────────────────
TASK_SLUG    = os.getenv("TASK_PICK_SLUG",  "pick-place-block")
TASK_DESC    = os.getenv("TASK_PICK_DESC",  "Pick up the red block and place it in the bowl")
REF_REPO     = f"{HF_USER}/soarm101-{TASK_SLUG}-reference"
DATASET_REPO = f"{HF_USER}/soarm101-{TASK_SLUG}"
MODEL_REPO   = f"{HF_USER}/{POLICY}-{TASK_SLUG}"

# Pick-and-place needs more episodes and time than touch
NUM_EPISODES = 80   # grasp is harder — more examples help
EPISODE_TIME = 30
RESET_TIME   = 12   # reset includes repositioning the block

_bench["config"] = {"task": TASK_SLUG, "dataset": DATASET_REPO,
                    "model": MODEL_REPO, "policy": POLICY}

print(f"Task:    {TASK_DESC}")
print(f"Dataset: {DATASET_REPO}")
print(f"Model:   {MODEL_REPO}")
print(f"Node:    {FLOATING_IP}")
total = NUM_EPISODES * (EPISODE_TIME + RESET_TIME) - RESET_TIME
print(f"Session: {NUM_EPISODES} episodes × {EPISODE_TIME}s  ≈ {total//60}m {total%60}s")

## 1. Scene Setup Check

Pick-and-place requires a consistent scene. Verify before every session.

In [ ]:
from IPython.display import IFrame, display

PREVIEW_PORT = 7860

import subprocess

def pi_run(cmd, capture=False, stream=False):
    ssh_prefix = [
        "ssh", "-p", PI_PORT, "-o", "StrictHostKeyChecking=no",
        "-o", "ConnectTimeout=10", f"{PI_USER}@{PI_HOST}",
    ]
    full = ssh_prefix + ["bash", "-c", cmd]
    if stream:
        proc = subprocess.Popen(full, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end="")
        return proc.wait()
    elif capture:
        r = subprocess.run(full, capture_output=True, text=True)
        return r.stdout.strip() if r.returncode == 0 else ""
    return subprocess.run(full, capture_output=True, text=True)

# Open camera preview
pi_run("balena stop $(balena ps -q 2>/dev/null) 2>/dev/null || true")
time.sleep(2)
pi_run(
    f"balena run -d --privileged --device=/dev/video0 --device=/dev/video2 "
    f"-p {PREVIEW_PORT}:{PREVIEW_PORT} {PI_IMAGE} python scripts/camera_preview.py"
)
time.sleep(3)

tunnel = subprocess.Popen(shlex.split(
    f"ssh -p {PI_PORT} -L {PREVIEW_PORT}:localhost:{PREVIEW_PORT} "
    f"-N -o StrictHostKeyChecking=no {PI_USER}@{PI_HOST}"
))
time.sleep(2)

print("Verify scene before starting:")
print("  - Block is in the START position (marked on table)")
print("  - Bowl is in the TARGET position (marked on table)")
print("  - Both visible in top camera, gripper visible in side camera")
print()
display(IFrame(src=f"http://localhost:{PREVIEW_PORT}", width="100%", height=480))

input("\nPress Enter when scene is set: ")
tunnel.terminate()
pi_run("balena stop $(balena ps -q 2>/dev/null) 2>/dev/null || true")

## 2. Replay Reference Animation

Watch the reference before coaching. Pay attention to grip timing and lift height.

In [ ]:
print(f"Playing reference animation from {REF_REPO}")
print("Note: where the gripper closes, how high it lifts, where it releases.")
print()
print(f"ssh -p {PI_PORT} -t {PI_USER}@{PI_HOST} \\")
print(f'  "balena run -it --privileged \\')
print(f'   --device=/dev/ttyACM1 \\')
print(f'   -v /mnt/data/calibration:/app/calibration \\')
print(f'   -v /mnt/data/datasets:/app/data \\')
print(f'   {PI_IMAGE} \\')
print(f'   lerobot-replay \\')
print(f'     --robot.type=so101_follower --robot.port=/dev/ttyACM1 \\')
print(f'     --robot.id=alpha_follower \\')
print(f'     --robot.calibration_dir=/app/calibration \\')
print(f'     --dataset.repo_id={REF_REPO} \\')
print(f'     --dataset.episode=0 \\')
print(f'     --play_sounds=false"')

input("\nPress Enter after watching the reference: ")

## 3. Coach the Robot — Pick and Place

**Coaching tips for pick-and-place**:
- Approach the block from directly above
- Close the gripper before making contact — not after
- Lift straight up at least 5cm before moving laterally
- Move deliberately to the bowl; don't rush the lateral motion
- Open gripper fully and hold 1s before retracting
- Return to the exact same home position after each episode
- **Reset**: replace the block in the start position before each episode

In [ ]:
print(f"Ready to coach: {TASK_DESC}")
print(f"Collecting {NUM_EPISODES} episodes × {EPISODE_TIME}s")
print()
print("Run in your terminal:")
print()
print(f"ssh -p {PI_PORT} -t {PI_USER}@{PI_HOST} \\")
print(f'  "balena run -it --privileged \\')
print(f'   --device=/dev/ttyACM0 --device=/dev/ttyACM1 \\')
print(f'   --device=/dev/video0 --device=/dev/video2 \\')
print(f'   -v /tmp/fleet.yaml:/app/config/fleet.yaml \\')
print(f'   -v /mnt/data/calibration:/app/calibration \\')
print(f'   -v /mnt/data/datasets:/app/data \\')
print(f'   -e HF_TOKEN={HF_TOKEN} \\')
print(f'   {PI_IMAGE} \\')
print(f'   lerobot-record \\')
print(f'     --robot.type=so101_follower --robot.port=/dev/ttyACM1 \\')
print(f'     --robot.calibration_dir=/app/calibration \\')
print(f'     --teleop.type=so101_leader --teleop.port=/dev/ttyACM0 \\')
print(f'     --teleop.calibration_dir=/app/calibration \\')
print(f'     --dataset.repo_id={DATASET_REPO} \\')
print(f'     --dataset.num_episodes={NUM_EPISODES} \\')
print(f"     --dataset.task='{TASK_DESC}' \\'")
print(f'     --dataset.push_to_hub=true"')

input("\nPress Enter when coaching session is complete: ")
print(f"Dataset: https://huggingface.co/datasets/{DATASET_REPO}")

## 4. Verify Episode Quality

In [ ]:
print("Replaying episode 0 — compare to reference.")
print("Check: gripper closes on block, lifts cleanly, block lands in bowl.")
print()
print(f"ssh -p {PI_PORT} -t {PI_USER}@{PI_HOST} \\")
print(f'  "balena run -it --privileged --device=/dev/ttyACM1 \\')
print(f'   -v /mnt/data/calibration:/app/calibration \\')
print(f'   -v /mnt/data/datasets:/app/data \\')
print(f'   {PI_IMAGE} \\')
print(f'   lerobot-replay \\')
print(f'     --robot.type=so101_follower --robot.port=/dev/ttyACM1 \\')
print(f'     --robot.id=alpha_follower \\')
print(f'     --robot.calibration_dir=/app/calibration \\')
print(f'     --dataset.repo_id={DATASET_REPO} \\')
print(f'     --dataset.episode=0 --play_sounds=false"')

quality = input("\nData quality OK? [y/N]: ")
if quality.strip().lower() not in ('y', 'yes'):
    print("Re-collect before training. Common issues:")
    print("  - Inconsistent start positions → mark the table")
    print("  - Gripper not closing fully → slow down the grasp")
    print("  - Block dropping mid-air → grip before lifting")
else:
    print("Quality confirmed — proceeding to training.")

## 5. Train on MI100

In [ ]:
_t_train = time.monotonic()

if not FLOATING_IP or FLOATING_IP == "REPLACE_ME_AFTER_PROVISION":
    raise RuntimeError("CONTROL_FLOATING_IP not set — run 01_reserve_node first")

# Pick-and-place benefits from more training steps than touch
train_cmd = (
    f"source ~/miniconda3/bin/activate lerobot && "
    f"cd ~/lerobot && "
    f"python lerobot/scripts/train.py "
    f"--dataset.repo_id={DATASET_REPO} "
    f"--policy.path=lerobot/{POLICY} "
    f"--output_dir=outputs/train/{POLICY}_{TASK_SLUG} "
    f"--job_name={POLICY}_{TASK_SLUG} "
    f"--policy.device=cuda "
    f"--wandb.enable=false"
)

launch = subprocess.run(
    ["ssh", "-o", "StrictHostKeyChecking=no", f"cc@{FLOATING_IP}",
     f"tmux kill-session -t train 2>/dev/null || true; "
     f"tmux new-session -d -s train '{train_cmd}' && echo launched"],
    capture_output=True, text=True
)
print(f"Training: {'launched' if 'launched' in launch.stdout else launch.stderr.strip()}")
print(f"  Monitor: ssh cc@{FLOATING_IP}  →  tmux attach -t train")

_bench["timings"]["training_start_s"] = round(time.monotonic() - _t_train, 1)

## 6. Monitor and Upload

In [ ]:
TOTAL_STEPS   = 100000  # pick-and-place needs more steps
POLL_INTERVAL = 60
LOG_PATH      = f"~/lerobot/outputs/train/{POLICY}_{TASK_SLUG}/train.log"

with tqdm(total=TOTAL_STEPS, unit="step", desc="Training") as pbar:
    last_step = 0
    for _ in range(120):
        out = subprocess.run(
            ["ssh", "-o", "StrictHostKeyChecking=no", f"cc@{FLOATING_IP}",
             f"tail -n 3 {LOG_PATH} 2>/dev/null"],
            capture_output=True, text=True
        ).stdout.strip()
        steps = re.findall(r'step=(\d+)', out)
        if steps:
            cur = int(steps[-1])
            pbar.update(cur - last_step)
            last_step = cur
        if out:
            pbar.set_postfix_str(out.splitlines()[-1][:60])
        if last_step >= TOTAL_STEPS:
            break
        time.sleep(POLL_INTERVAL)

print("\nUploading checkpoint...")
up = subprocess.run(
    ["ssh", "-o", "StrictHostKeyChecking=no", f"cc@{FLOATING_IP}",
     f"HF_TOKEN={HF_TOKEN} huggingface-cli upload {MODEL_REPO} "
     f"~/lerobot/outputs/train/{POLICY}_{TASK_SLUG}/checkpoints/last/ ."],
    capture_output=True, text=True
)
print(f"Checkpoint: {'https://huggingface.co/' + MODEL_REPO if up.returncode == 0 else up.stderr.strip()}")

_bench["timings"]["total_s"] = round(time.monotonic() - _t0, 1)
_bench["completed_at"] = datetime.utcnow().isoformat()

results_dir = Path("..") / "bench" / "results"
results_dir.mkdir(exist_ok=True)
ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
bench_path = results_dir / f"06_pick_place_{ts}.json"
bench_path.write_text(json.dumps(_bench, indent=2))
print(json.dumps(_bench, indent=2))
print(f"\nBenchmark saved: {bench_path}")